# 01 - Scrape Tanqeeb Job Listings

This notebook is the **scraping** stage of the Tanqeeb pipeline.

- **Step 1 - Discover:** collects job URLs from the Tanqeeb search results (Selenium, since results are JS-rendered), starting at page 1 and stopping once it catches up with jobs already scraped.
- **Step 2 - Scrape:** fetches each new job page (`requests` + `BeautifulSoup`) **in batches**. After every batch, the data is written to disk.
- **Step 3 - Summary.**

### Crash-safe / resumable
If the notebook fails at any point (network error, Chrome crash, Ctrl-C, ...), just **re-run all cells**:

| File | What it holds | Purpose |
|---|---|---|
| `tanqeeb_scraped_urls_checkpoint.json` | **URLs only** | Every URL already scraped -> never scraped twice |
| `tanqeeb_new_urls_checkpoint.json` | URLs still to scrape (this run) | Resume mid-run; skips pagination if discovery already finished |
| `tanqeeb_jobs_raw_<run_id>.json` | Jobs scraped in this run | Audit copy (same file is reused when a run is resumed) |
| `tanqeeb_pending_raw_queue.json` | Jobs waiting for `02_clean_Tanqeeb.ipynb` | Hand-off queue (append-only until cleaned) |

In [ ]:
import json
import os
import time
from datetime import datetime
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter
from urllib3.exceptions import ReadTimeoutError
from urllib3.util.retry import Retry

from selenium import webdriver
from selenium.common.exceptions import TimeoutException, WebDriverException
from selenium.webdriver.chrome.options import Options

In [ ]:
# ============================================================
# CONFIG
# ============================================================

BASE_URL = "https://saudi.tanqeeb.com"
SEARCH_PATH = f"{BASE_URL}/jobs/search"
QUERY_STRING = (
    "keywords=&country=54&state=0&category=-1&"
    "workplace=0&search_period=0&lang=all&change_lang=1"
)
SOURCE_NAME = "Tanqeeb"

# ---- Files ---------------------------------------------------
RAW_DIR = "../data/raw"

# URL-ONLY checkpoint: a plain JSON list of every job URL ever scraped.
# This is the source of truth for "do we already have this job?".
URLS_CHECKPOINT = f"{RAW_DIR}/tanqeeb_scraped_urls_checkpoint.json"

# Old full-job checkpoint from previous versions of this notebook.
# If it exists and the URL checkpoint doesn't, URLs are migrated from it once.
LEGACY_JOBS_CHECKPOINT = f"{RAW_DIR}/tanqeeb_jobs_checkpoint.json"

# State of the CURRENT run: URLs discovered but not scraped yet.
# Deleted automatically when the run finishes.
NEW_URLS_CHECKPOINT = f"{RAW_DIR}/tanqeeb_new_urls_checkpoint.json"

# Hand-off queue for 02_clean_Tanqeeb.ipynb (scraped, not yet cleaned).
PENDING_RAW_QUEUE = f"{RAW_DIR}/tanqeeb_pending_raw_queue.json"

# ---- Batching ------------------------------------------------
# Everything is flushed to disk after this many URLs are handled.
BATCH_SIZE = 50

# ---- Discovery (Selenium) -----------------------------------
INCREMENTAL_MODE = True                  # only queue URLs not already scraped
STOP_AFTER_CONSECUTIVE_KNOWN_PAGES = 2   # stop paginating once we've caught up
MAX_PAGES = 1                            # NOTE: ~1,800+ pages exist; raise for a real run (e.g. 2000)
MAX_CONSECUTIVE_FAILURES = 5             # driver failures in a row before giving up
PAGE_LOAD_WAIT = 5                       # seconds to let JS render each search page

# ---- Scraping (requests) ------------------------------------
REQUEST_TIMEOUT = 30
REQUEST_DELAY = 1                        # seconds between job-page requests
REQUEST_HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/140.0.0.0 Safari/537.36"
    )
}

In [ ]:
# ============================================================
# HELPER FUNCTIONS
# ============================================================

# ---------- JSON I/O (atomic: a crash mid-write can't corrupt a file) ----------

def save_json(path, data):
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    tmp_path = path + ".tmp"
    with open(tmp_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    os.replace(tmp_path, path)          # atomic swap


def load_json(path, default):
    if not os.path.exists(path):
        return default
    try:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    except json.JSONDecodeError:
        print(f"  !! {path} is corrupted/unreadable -- ignoring it.")
        return default


def merge_jobs_by_url(existing, new_jobs):
    """Merge job lists keyed by URL (new versions replace old, order preserved)."""
    merged = {j.get("url"): j for j in existing}
    for j in new_jobs:
        merged[j.get("url")] = j
    return list(merged.values())


# ---------- URL-only checkpoint ----------

def load_url_checkpoint():
    """Set of every URL already scraped. Migrates from the legacy full-job file once."""
    if os.path.exists(URLS_CHECKPOINT):
        return set(load_json(URLS_CHECKPOINT, []))

    if os.path.exists(LEGACY_JOBS_CHECKPOINT):
        legacy_jobs = load_json(LEGACY_JOBS_CHECKPOINT, [])
        urls = {j.get("url") for j in legacy_jobs
                if j.get("url") and j.get("url") != "N/A"}
        save_json(URLS_CHECKPOINT, sorted(urls))
        print(f"Migrated {len(urls)} URLs from the legacy jobs checkpoint "
              f"({LEGACY_JOBS_CHECKPOINT}). It is no longer used and can be deleted.")
        return urls

    return set()


def save_url_checkpoint(urls):
    save_json(URLS_CHECKPOINT, sorted(urls))


# ---------- Current-run state ----------

def load_run_state():
    """
    {run_id, discovery_complete, new_urls}. A leftover file means the last run
    was interrupted; we resume from it.
    """
    state = {"run_id": None, "discovery_complete": False, "new_urls": []}
    state.update(load_json(NEW_URLS_CHECKPOINT, {}))
    if not state["run_id"]:
        state["run_id"] = datetime.now().strftime("%Y%m%d_%H%M%S")
    return state


def save_run_state(state):
    save_json(NEW_URLS_CHECKPOINT, state)


def clear_run_state():
    if os.path.exists(NEW_URLS_CHECKPOINT):
        os.remove(NEW_URLS_CHECKPOINT)


# ---------- Discovery helpers ----------

def build_page_url(page_num):
    """Page 1: /jobs/search?...   Page N: /jobs/search/page/N?..."""
    if page_num <= 1:
        return f"{SEARCH_PATH}?{QUERY_STRING}"
    return f"{SEARCH_PATH}/page/{page_num}?{QUERY_STRING}"


def get_job_links_from_html(html):
    soup = BeautifulSoup(html, "html.parser")
    return soup.select("h2.search-job-title a.search-job-title-link")


def make_driver():
    """Fresh headless Chrome with a longer command timeout."""
    options = Options()
    options.add_argument("--headless=new")
    options.add_argument("--window-size=1920,1080")
    options.add_argument("--disable-blink-features=AutomationControlled")
    driver = webdriver.Chrome(options=options)
    driver.command_executor.set_timeout(300)
    return driver


# ---------- Scraping helpers ----------

def make_session():
    """requests session with automatic retries + backoff on transient errors."""
    retry = Retry(
        total=3,
        backoff_factor=2,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"],
    )
    session = requests.Session()
    session.mount("https://", HTTPAdapter(max_retries=retry))
    session.mount("http://", HTTPAdapter(max_retries=retry))
    session.headers.update(REQUEST_HEADERS)
    return session


def _text(element):
    return element.get_text(" ", strip=True) if element else None


def _abs_url(element, attr):
    value = element.get(attr) if element else None
    return urljoin(BASE_URL, value) if value else None


WORK_MODES = {"on-site", "remote", "hybrid"}
JOB_TYPES = {"full time", "part time", "contract", "internship", "temporary", "freelance"}


def parse_job_page(html, url):
    """Turn one job page's HTML into a job dict."""
    soup = BeautifulSoup(html, "html.parser")

    # Apply button -> job id + apply url
    apply_button = soup.select_one(".apply-btn")
    job_id = apply_button.get("data-job-id") if apply_button else None
    apply_url = (apply_button.get("href") or apply_button.get("data-url")) if apply_button else None

    # Company
    company_el = soup.select_one(
        "a.job-meta-company, .job-company-name, .company-name, a.company-name-link"
    )
    company_logo_el = soup.select_one(".company-logo img, .job-company-logo img")

    # Tags -> work mode / job type
    tags = [t.get_text(" ", strip=True)
            for t in soup.select(".job-tags .job-tag") if t.get_text(strip=True)]
    work_mode = job_type = None
    for tag in tags:
        if tag.lower() in WORK_MODES:
            work_mode = tag
        elif tag.lower() in JOB_TYPES:
            job_type = tag

    # Posted date
    date_el = soup.select_one(".job-date")
    parsed_date = pd.to_datetime(date_el.get("data-datetime") if date_el else None,
                                 errors="coerce")
    posted_date = None if pd.isna(parsed_date) else parsed_date.isoformat()

    # Every label/value row in the details block (kept generic on purpose)
    details = {}
    for row in soup.select(".meta-data .meta"):
        spans = row.find_all("span")
        if len(spans) >= 2:
            details[spans[0].get_text(" ", strip=True)] = spans[1].get_text(" ", strip=True)

    skills_list = [s.get_text(" ", strip=True)
                   for s in soup.select(".job-skills .skill, .skills-list .skill-tag")
                   if s.get_text(strip=True)]

    return {
        "id": job_id,
        "source": SOURCE_NAME,
        "title": _text(soup.select_one("h3.job-title-text")),
        "company": _text(company_el),
        "company_url": _abs_url(company_el, "href"),
        "company_logo_url": _abs_url(company_logo_el, "src"),
        "description": _text(soup.select_one("#jobDescriptionBody")),
        "url": url,
        "apply_url": apply_url,
        "location": _text(soup.select_one(".job-meta-item span")),
        "work_mode": work_mode,
        "job_type": details.get("Employment") or job_type,
        "career_level": details.get("Career Level"),
        "experience": details.get("Experience"),
        "education": details.get("Education"),
        "gender": details.get("Gender"),
        "industry": details.get("Industry"),
        "skills": "; ".join(skills_list) if skills_list else None,
        "salary": details.get("Salary"),
        "posted_date": posted_date,
        "collected_at": datetime.now().isoformat(),
    }

## Step 1: Load checkpoints, then discover NEW job URLs

In [ ]:
scraped_urls = load_url_checkpoint()      # URLs only
state = load_run_state()                  # leftover state = interrupted run
RAW_JSON_OUTPUT = f"{RAW_DIR}/tanqeeb_jobs_raw_{state['run_id']}.json"

print(f"URLs already scraped (all previous runs): {len(scraped_urls)}")
print(f"Run id: {state['run_id']}")
print("Incremental mode:", "ON" if INCREMENTAL_MODE else "OFF (full re-scrape)")

if state["new_urls"]:
    print(f"Resuming an interrupted run: {len(state['new_urls'])} URL(s) still to scrape.")


def discover_new_urls(state, known_urls):
    """
    Paginate from page 1 and queue unseen job URLs into state['new_urls'].
    State is saved after every page. Returns True if discovery finished
    cleanly (False if it gave up after repeated driver failures).
    """
    new_urls = state["new_urls"]
    queued = set(new_urls)
    driver = make_driver()
    failures = 0
    known_pages = 0
    page_num = 1
    finished = False

    try:
        while page_num <= MAX_PAGES:
            page_url = build_page_url(page_num)
            print(f"\nLoading search page {page_num}: {page_url}")

            try:
                driver.get(page_url)
                time.sleep(PAGE_LOAD_WAIT)
                html = driver.page_source
                failures = 0
            except (TimeoutException, WebDriverException, ReadTimeoutError) as e:
                failures += 1
                print(f"  !! Driver error on page {page_num} "
                      f"(failure {failures}/{MAX_CONSECUTIVE_FAILURES}): {e}")
                try:
                    driver.quit()
                except Exception:
                    pass
                if failures >= MAX_CONSECUTIVE_FAILURES:
                    print("Too many consecutive driver failures. Stopping pagination "
                          "(progress so far is saved).")
                    break
                time.sleep(3)
                driver = make_driver()
                continue                     # retry same page with a fresh browser

            job_links = get_job_links_from_html(html)
            print(f"  -> Job links found on this page: {len(job_links)}")

            if not job_links:
                print("No more job listings found. Stopping pagination.")
                finished = True
                break

            new_on_page = 0
            for link in job_links:
                href = link.get("href")
                if not href:
                    continue
                url = urljoin(BASE_URL, href)
                if INCREMENTAL_MODE and url in known_urls:
                    continue
                if url not in queued:
                    queued.add(url)
                    new_urls.append(url)
                    new_on_page += 1

            if new_on_page:
                known_pages = 0
                print(f"  -> New jobs found on this page: {new_on_page}")
            else:
                known_pages += 1
                print(f"  -> No new jobs on this page "
                      f"({known_pages}/{STOP_AFTER_CONSECUTIVE_KNOWN_PAGES}).")
                # incremental: caught up with old jobs; full mode: repeated last page
                if not INCREMENTAL_MODE or known_pages >= STOP_AFTER_CONSECUTIVE_KNOWN_PAGES:
                    print("Caught up / reached the end. Stopping pagination.")
                    finished = True
                    break

            save_run_state(state)            # persist discovery progress every page
            page_num += 1
        else:
            finished = True                  # reached MAX_PAGES
    finally:
        try:
            driver.quit()
        except Exception:
            pass

    state["discovery_complete"] = finished
    save_run_state(state)
    return finished


if state["discovery_complete"] and state["new_urls"]:
    print("Discovery already completed in the interrupted run -- skipping pagination.")
else:
    discover_new_urls(state, scraped_urls)

print(f"\nJob URLs to scrape this run: {len(state['new_urls'])}")

## Step 2: Scrape new job pages -- saving to disk after every batch

In [ ]:
session = make_session()

# If this run was resumed, keep the jobs that earlier attempts already saved.
run_jobs = load_json(RAW_JSON_OUTPUT, [])
pending_queue = load_json(PENDING_RAW_QUEUE, [])

todo = [u for u in state["new_urls"]
        if not (INCREMENTAL_MODE and u in scraped_urls)]
print(f"{len(todo)} URL(s) to scrape, saving every {BATCH_SIZE}.")

batch_jobs = []      # scraped jobs not yet flushed to disk
batch_done = []      # URLs finished (scraped, or permanently gone) not yet flushed
failed_urls = []     # temporary failures -- NOT marked as scraped, retried next run
stats = {"scraped": 0, "gone": 0, "failed": 0}


def flush_batch():
    """
    Persist the current batch. Order matters: data files first, URL checkpoint
    last -- so a crash in the middle can never mark a URL as done without its
    job data being saved (worst case: it is scraped again and de-duplicated).
    """
    global run_jobs, pending_queue, batch_jobs, batch_done

    if not batch_done:
        return

    run_jobs = merge_jobs_by_url(run_jobs, batch_jobs)
    save_json(RAW_JSON_OUTPUT, run_jobs)                     # audit copy of this run

    pending_queue = merge_jobs_by_url(pending_queue, batch_jobs)
    save_json(PENDING_RAW_QUEUE, pending_queue)              # hand-off to 02_clean

    scraped_urls.update(batch_done)
    save_url_checkpoint(scraped_urls)                        # URL-only checkpoint

    done = set(batch_done)
    state["new_urls"] = [u for u in state["new_urls"] if u not in done]
    save_run_state(state)                                    # remaining work

    print(f"  [saved] batch of {len(batch_done)} URLs "
          f"({len(batch_jobs)} jobs) | {len(scraped_urls)} URLs checkpointed | "
          f"{len(state['new_urls'])} left")
    batch_jobs, batch_done = [], []


try:
    for i, url in enumerate(todo, start=1):
        print(f"[{i}/{len(todo)}] {url}")

        try:
            response = session.get(url, timeout=REQUEST_TIMEOUT)
        except requests.RequestException as e:
            print(f"  Request error: {e}")
            failed_urls.append(url)
            stats["failed"] += 1
            continue

        if response.status_code in (404, 410):
            # Listing was removed -- mark done so it isn't rediscovered forever.
            print(f"  Gone ({response.status_code}), skipping permanently.")
            batch_done.append(url)
            stats["gone"] += 1
        elif response.status_code != 200:
            print(f"  Failed with status {response.status_code} (will retry next run).")
            failed_urls.append(url)
            stats["failed"] += 1
            continue
        else:
            try:
                job = parse_job_page(response.text, url)
            except Exception as e:
                print(f"  Parse error: {e} (will retry next run).")
                failed_urls.append(url)
                stats["failed"] += 1
                continue
            batch_jobs.append(job)
            batch_done.append(url)
            stats["scraped"] += 1
            print(f"  Scraped: {job['title']}")

        if len(batch_done) >= BATCH_SIZE:
            flush_batch()

        time.sleep(REQUEST_DELAY)

finally:
    # Runs on normal completion AND on any error / Ctrl-C / kernel interrupt:
    # the partial batch is saved, so re-running resumes right where it stopped.
    flush_batch()

# Reached only if the loop completed without crashing. Failed URLs were never
# added to the URL checkpoint, so the next run's pagination will find them again.
clear_run_state()

## Step 3: Summary

In [ ]:
print("=" * 60)
print(f"Scraped this run:              {stats['scraped']}")
print(f"Removed listings (404/410):    {stats['gone']}")
print(f"Failed (retry on next run):    {stats['failed']}")
print(f"Total URLs in checkpoint:      {len(scraped_urls)}")
print(f"Jobs waiting to be cleaned:    {len(pending_queue)}")
print(f"This run's raw copy:           {RAW_JSON_OUTPUT}")
print(f"Pending queue for 02_clean:    {PENDING_RAW_QUEUE}")
print(f"URL checkpoint:                {URLS_CHECKPOINT}")
print("=" * 60)